
# ADVANCED DATA CLEANING & FEATURE ENGINEERING


In [4]:
import pandas as pd
import numpy as np

In [7]:
# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_excel("../dataset/online_retail_II.xlsx")

In [8]:
clean_df = df.copy()

print(f"Raw dataset shape: {clean_df.shape}")

Raw dataset shape: (525461, 8)


In [9]:
# ============================================================
# 1. STANDARDIZE COLUMN NAMES
# ============================================================

clean_df = clean_df.rename(columns={
    "Invoice": "Invoice_ID",
    "StockCode": "Stock_Code",
    "Description": "Description",
    "Quantity": "Quantity",
    "InvoiceDate": "Invoice_Date",
    "Price": "Unit_Price",
    "Customer ID": "Customer_ID",
    "Country": "Country"
})

print(clean_df.columns.tolist())

['Invoice_ID', 'Stock_Code', 'Description', 'Quantity', 'Invoice_Date', 'Unit_Price', 'Customer_ID', 'Country']


In [10]:
# ============================================================
# 2. TEXT STANDARDIZATION
# ============================================================

clean_df["Invoice_ID"] = clean_df["Invoice_ID"].astype(str).str.strip()
clean_df["Stock_Code"] = clean_df["Stock_Code"].astype(str).str.strip()
clean_df["Country"] = clean_df["Country"].astype(str).str.strip()

# Clean product descriptions
clean_df["Description"] = (
    clean_df["Description"]
    .astype("string")
    .str.strip()
    .str.upper()
)

print("Text standardization completed.")

Text standardization completed.


In [11]:
# ============================================================
# 3. DUPLICATE ANALYSIS
# ============================================================

duplicate_count = clean_df.duplicated().sum()

print(f"Duplicate rows before cleaning: {duplicate_count:,}")

Duplicate rows before cleaning: 6,865


In [12]:
clean_df = clean_df.drop_duplicates().copy()

print(f"Shape after removing exact duplicates: {clean_df.shape}")

Shape after removing exact duplicates: (518596, 8)


In [13]:
# ============================================================
# 4. RETURN / NEGATIVE TRANSACTION IDENTIFICATION
# ============================================================

clean_df["Is_Return"] = np.where(
    clean_df["Quantity"] < 0,
    1,
    0
)

print(
    f"Return/negative quantity rows: {clean_df['Is_Return'].sum():,}"
)

Return/negative quantity rows: 12,302


In [15]:
return_transactions = clean_df[
    clean_df["Is_Return"] == 1
]

display(
    return_transactions[
        [
            "Invoice_ID",
            "Stock_Code",
            "Description",
            "Quantity",
            "Invoice_Date",
            "Unit_Price",
            "Customer_ID"
        ]
    ].head(10)
)

,Invoice_ID,Stock_Code,Description,Quantity,Invoice_Date,Unit_Price,Customer_ID
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0


In [16]:
# ============================================================
# 5. SALES-ONLY DATASET
# ============================================================

sales_df = clean_df[
    (clean_df["Quantity"] > 0) &
    (clean_df["Unit_Price"] > 0)
].copy()

print(f"Sales dataset shape: {sales_df.shape}")

Sales dataset shape: (504731, 9)


In [18]:
# ============================================================
# 6. PRICE VALIDATION
# ============================================================

invalid_price_rows = clean_df[
    clean_df["Unit_Price"] <= 0
]

print(f"Invalid price rows: {len(invalid_price_rows):,}")

display(
    invalid_price_rows[
        [
            "Invoice_ID",
            "Stock_Code",
            "Description",
            "Quantity",
            "Unit_Price"
        ]
    ].head(10)
)

Invalid price rows: 3,684


,Invoice_ID,Stock_Code,Description,Quantity,Unit_Price
263,489464,21733,85123A MIXED,-96,0.0
283,489463,71477,SHORT,-240,0.0
284,489467,85123A,21733 MIXED,-192,0.0
470,489521,21646,<NA>,-50,0.0
3114,489655,20683,<NA>,-44,0.0
3161,489659,21350,<NA>,230,0.0
3162,489660,35956,LOST,-1043,0.0
3168,489663,35605A,DAMAGES,-117,0.0
3731,489781,84292,<NA>,17,0.0
4296,489806,18010,<NA>,-770,0.0


In [21]:
# ============================================================
# 7. RECOVER MISSING PRODUCT DESCRIPTIONS
# ============================================================

description_mapping = (
    clean_df
    .dropna(subset=["Description"])
    .groupby("Stock_Code")["Description"]
    .first()
)

clean_df["Description"] = (
    clean_df["Description"]
    .fillna(clean_df["Stock_Code"].map(description_mapping))
)

remaining_missing_descriptions = clean_df["Description"].isna().sum()

print(
    f"Missing descriptions after recovery: {remaining_missing_descriptions:,} "
)

Missing descriptions after recovery: 365 


In [22]:
# ============================================================
# 8. CUSTOMER ID VALIDATION
# ============================================================

print(
    f"Missing Customer IDs: "
    f"{sales_df['Customer_ID'].isna().sum():,}"
)

Missing Customer IDs: 103,815


In [23]:
# ============================================================
# 9. REVENUE FEATURE
# ============================================================

sales_df["Revenue"] = (
    sales_df["Quantity"] *
    sales_df["Unit_Price"]
)

display(
    sales_df[
        [
            "Invoice_ID",
            "Stock_Code",
            "Quantity",
            "Unit_Price",
            "Revenue"
        ]
    ].head()
)

,Invoice_ID,Stock_Code,Quantity,Unit_Price,Revenue
0,489434,85048,12,6.95,83.4
1,489434,79323P,12,6.75,81.0
2,489434,79323W,12,6.75,81.0
3,489434,22041,48,2.10,100.8
4,489434,21232,24,1.25,30.0


In [24]:
# ============================================================
# 10. DATE FEATURES
# ============================================================

sales_df["Year"] = sales_df["Invoice_Date"].dt.year
sales_df["Month"] = sales_df["Invoice_Date"].dt.month
sales_df["Month_Name"] = sales_df["Invoice_Date"].dt.month_name()
sales_df["Quarter"] = sales_df["Invoice_Date"].dt.quarter
sales_df["Day"] = sales_df["Invoice_Date"].dt.day
sales_df["Day_of_Week"] = sales_df["Invoice_Date"].dt.day_name()
sales_df["Hour"] = sales_df["Invoice_Date"].dt.hour

display(
    sales_df[
        [
            "Invoice_Date",
            "Year",
            "Month",
            "Month_Name",
            "Quarter",
            "Day_of_Week",
            "Hour"
        ]
    ].head()
)

,Invoice_Date,Year,Month,Month_Name,Quarter,Day_of_Week,Hour
0,2009-12-01 07:45:00,2009,12,December,4,Tuesday,7
1,2009-12-01 07:45:00,2009,12,December,4,Tuesday,7
2,2009-12-01 07:45:00,2009,12,December,4,Tuesday,7
3,2009-12-01 07:45:00,2009,12,December,4,Tuesday,7
4,2009-12-01 07:45:00,2009,12,December,4,Tuesday,7


In [27]:
# ============================================================
# 11. PRICE BANDS
# ============================================================

sales_df["Price_Band"] = pd.cut(
    sales_df["Unit_Price"],
    bins=[0, 1, 2, 5, 10, 25, 50, np.inf],
    labels=[
        "<=1",
        "1-2",
        "2-5",
        "5-10",
        "10-25",
        "25-50",
        "50+"
    ],
    include_lowest=True
)

display(
    sales_df["Price_Band"].value_counts()
    .sort_index()
)

Price_Band
<=1       93140
1-2      143138
2-5      173665
5-10      69478
10-25     21768
25-50      2572
50+         970
Name: count, dtype: int64

In [28]:
# ============================================================
# 12. ORDER VALUE
# ============================================================

order_value = (
    sales_df
    .groupby("Invoice_ID")["Revenue"]
    .sum()
    .rename("Order_Value")
)

sales_df = sales_df.merge(
    order_value,
    on="Invoice_ID",
    how="left"
)

display(
    sales_df[
        ["Invoice_ID", "Revenue", "Order_Value"]
    ].head(10)
)

,Invoice_ID,Revenue,Order_Value
0,489434,83.4,505.3
1,489434,81.0,505.3
2,489434,81.0,505.3
3,489434,100.8,505.3
4,489434,30.0,505.3
5,489434,39.6,505.3
6,489434,30.0,505.3
7,489434,59.5,505.3
8,489435,30.6,145.8
9,489435,45.0,145.8


In [29]:
# ============================================================
# 13. CUSTOMER ANALYSIS DATASET
# ============================================================

customer_df = sales_df[
    sales_df["Customer_ID"].notna()
].copy()

print(f"Customer analysis rows: {len(customer_df):,}")
print(
    f"Unique customers: "
    f"{customer_df['Customer_ID'].nunique():,}"
)

Customer analysis rows: 400,916
Unique customers: 4,312


In [30]:
# ============================================================
# 14. BASKET ANALYSIS DATASET
# ============================================================

basket_df = sales_df[
    [
        "Invoice_ID",
        "Stock_Code",
        "Description",
        "Quantity",
        "Unit_Price",
        "Revenue"
    ]
].copy()

print(f"Basket analysis rows: {len(basket_df):,}")
print(
    f"Unique invoices: "
    f"{basket_df['Invoice_ID'].nunique():,}"
)

Basket analysis rows: 504,731
Unique invoices: 20,952


In [31]:
# ============================================================
# 15. FINAL CLEANING CHECK
# ============================================================

print("========== FINAL DATA QUALITY CHECK ==========")

print(f"Sales rows              : {len(sales_df):,}")
print(f"Sales columns           : {sales_df.shape[1]}")
print(f"Negative quantities     : {(sales_df['Quantity'] < 0).sum():,}")
print(f"Invalid prices          : {(sales_df['Unit_Price'] <= 0).sum():,}")
print(f"Missing dates           : {sales_df['Invoice_Date'].isna().sum():,}")
print(f"Missing descriptions    : {sales_df['Description'].isna().sum():,}")
print(f"Unique products         : {sales_df['Stock_Code'].nunique():,}")
print(f"Unique invoices         : {sales_df['Invoice_ID'].nunique():,}")
print(f"Unique customers        : {sales_df['Customer_ID'].nunique():,}")

========== FINAL DATA QUALITY CHECK ==========
Sales rows              : 504,731
Sales columns           : 19
Negative quantities     : 0
Invalid prices          : 0
Missing dates           : 0
Missing descriptions    : 0
Unique products         : 4,250
Unique invoices         : 20,952
Unique customers        : 4,312


In [32]:
# ============================================================
# 16. SAVE CLEAN DATASETS
# ============================================================

sales_df.to_csv(
    "pricing_sales_clean.csv",
    index=False
)

customer_df.to_csv(
    "customer_analysis_clean.csv",
    index=False
)

basket_df.to_csv(
    "basket_analysis_clean.csv",
    index=False
)

print("\nAll cleaned datasets saved successfully.")


All cleaned datasets saved successfully.
